# Install the Require Libraries

In [ ]:
!pip install -qU langchain chromadb huggingface_hub sentence-transformers pypdf openai tiktoken

In [ ]:
!pip install -U langchain-community

# Let's Load the Data Now...

In [1]:
from langchain_community.document_loaders import PyPDFLoader

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_928\4175148793.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [4]:
import os
print(os.getcwd())

f:\Advance RAG\Multi-Source-RAG-with-MergerRetriever


In [5]:
loader_harrypotter  = PyPDFLoader(
    r"F:\Advance RAG\Data\harry_potter_book.pdf"
)

In [6]:
document_harrypotter = loader_harrypotter.load()

In [7]:
print(len(document_harrypotter))

6


In [8]:
loader_got = PyPDFLoader(r"F:\Advance RAG\Data\got_book.pdf")

In [9]:
document_got = loader_got.load()

In [10]:
print(len(document_got))


8


# Let's create the text splitter for chunking

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [12]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)

In [13]:
text_harrypotter = text_splitter.split_documents(document_harrypotter)

In [14]:
text_got = text_splitter.split_documents(document_got)

In [16]:
print(len(text_harrypotter))

69


In [17]:
print(len(text_got))

46


# Load the Embedding Model to Conver the Data into Vector

In [18]:
from langchain_community.embeddings import (
    HuggingFaceEmbeddings,
    HuggingFaceBgeEmbeddings,
)

In [19]:
# Initialize the Hugging Face embedding model.
# Uses the all-MiniLM-L6-v2 model to convert text into dense vector embeddings
# for semantic search, retrieval, and RAG applications.
minilm_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_928\2563379630.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  minilm_embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [20]:
# Initialize the BGE (BAAI General Embedding) model.
# Uses the bge-large-en model to generate high-quality dense vector embeddings
# optimized for semantic search, retrieval, and RAG applications.
bge_embeddings = HuggingFaceBgeEmbeddings(
    model_name="BAAI/bge-large-en"
)


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_928\3967482624.py:4: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  bge_embeddings = HuggingFaceBgeEmbeddings(


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [ ]:
os.environ["OPENAI_API_KEY"]=OPENAI_API_KEY

In [ ]:
openai_embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

# Now ingest the Data into the Chroma Database

In [21]:
from langchain_community.vectorstores import Chroma
import chromadb

In [22]:
CURRENT_DIR = os.getcwd()  #Get the current project directory
print(CURRENT_DIR)

f:\Advance RAG\Multi-Source-RAG-with-MergerRetriever


In [23]:
DB_DIR = os.path.join(CURRENT_DIR, "db") #Create a folder for the Chroma database
print(DB_DIR)

f:\Advance RAG\Multi-Source-RAG-with-MergerRetriever\db


In [24]:
client_settings = chromadb.config.Settings(  #Configure Chroma
    is_persistent=True,
    persist_directory=DB_DIR,
    anonymized_telemetry=False,
)

In [25]:
harrypotter_vectorstore = Chroma.from_documents(
    documents=text_harrypotter,
    embedding=bge_embeddings,
    client_settings=client_settings,
    collection_name="harrypotter",
    collection_metadata={"hnsw": "cosine"},
    persist_directory=os.path.join(DB_DIR, "harrypotter"),
)

In [26]:
got_vectorstore = Chroma.from_documents(
    documents=text_got,
    embedding=bge_embeddings,
    client_settings=client_settings,
    collection_name="got",
    collection_metadata={"hnsw": "cosine"},
    persist_directory=os.path.join(DB_DIR, "got"),
)

 # Now Create a Retriever

In [27]:
retriever_harrypotter = harrypotter_vectorstore.as_retriever(search_type="mmr",search_kwargs={"k": 5, "include_metadata": True})

In [29]:
result1 = retriever_harrypotter.invoke("Who is Harry Potter?")
result1

[Document(metadata={'title': '书籍1.indb', 'producer': 'Adobe PDF Library 10.0.1', 'total_pages': 6, 'gts_pdfxversion': 'PDF/X-4', 'page_label': '1', 'source': 'F:\\Advance RAG\\Data\\harry_potter_book.pdf', 'page': 0, 'creationdate': '2014-01-04T16:40:46+08:00', 'creator': 'Adobe InDesign CS6 (Windows)', 'moddate': '2014-01-04T17:23:49+08:00', 'trapped': '/False'}, page_content='DOI: http://dx.doi.org/10.3968/j.sll.1923156320130703.3020\nINTRODUCTION\nThe book series Harry Potter enjoy a global popularity \nin recent years. By now most readers are aware of what \nhas come to be called the Harry Potter phenomenon. The \nsudden success of the book has resulted in the translations \nof 67 languages. The book is featured with vivid language \nand the wonderful magic world it presents. And both \nchildren and adults are fascinated by Harry Potter and his \nunusual stories.'),
 Document(metadata={'page': 0, 'moddate': '2014-01-04T17:23:49+08:00', 'creationdate': '2014-01-04T16:40:46+08:00', '

In [30]:
for i, doc in enumerate(result1, start=1):
    print(f"\n{'='*60}")
    print(f"Document {i}")
    print(f"{'='*60}")
    print(doc.page_content)


Document 1
DOI: http://dx.doi.org/10.3968/j.sll.1923156320130703.3020
INTRODUCTION
The book series Harry Potter enjoy a global popularity 
in recent years. By now most readers are aware of what 
has come to be called the Harry Potter phenomenon. The 
sudden success of the book has resulted in the translations 
of 67 languages. The book is featured with vivid language 
and the wonderful magic world it presents. And both 
children and adults are fascinated by Harry Potter and his 
unusual stories.

Document 2
Education: Study on English Children’s Literature Translation After 
1979 (No. Y201223249).
Received 11 September 2013; accepted 6 December 2013
Abstract
Magic spells, being very specific in the  Harry Porter  
series, become an important part for translation studies.  
This paper aims to study the spell translation in Harry 
Potter from the perspective of Skopostheorie, a translation 
theory proposed by Hans Vemeer in the late 1970s. 
Based on Skopostheorie, this paper gives a com

In [31]:
retriever_got = got_vectorstore.as_retriever(search_type="mmr",search_kwargs={"k": 5, "include_metadata": True})

In [32]:
result2 = retriever_got.invoke("Who is Ser Waymar ?")
result2

[Document(metadata={'source': 'F:\\Advance RAG\\Data\\got_book.pdf', 'page_label': '2', 'producer': 'pypdf', 'creationdate': '', 'total_pages': 8, 'creator': 'PyPDF', 'page': 1}, page_content='with your commander.\nEspecially not a commander like this one.\nSer Waymar Royce was the youngest son of an ancient house with too many heirs. He \nwas a handsome youth of eighteen, grey-eyed and graceful and slender as a knife. \nMounted on his huge black destrier, the knight towered above Will and Gared on their \nsmaller garrons. He wore black leather boots, black woolen pants, black moleskin gloves, \nand a fine supple coat of gleaming black ringmail over layers of black wool and boiled'),
 Document(metadata={'producer': 'pypdf', 'total_pages': 8, 'page': 1, 'page_label': '2', 'creator': 'PyPDF', 'creationdate': '', 'source': 'F:\\Advance RAG\\Data\\got_book.pdf'}, page_content='and a fine supple coat of gleaming black ringmail over layers of black wool and boiled \nleather. Ser Waymar had b

# Let's Merge both Retriever

# It is also called lord of retriever(LOTR)

In [33]:
from langchain_classic.retrievers import MergerRetriever

In [34]:
lotr = MergerRetriever(retrievers=[retriever_harrypotter, retriever_got])

In [36]:
for chunk in lotr.invoke("Who is Harry Potter?"):
    print(chunk.page_content)

DOI: http://dx.doi.org/10.3968/j.sll.1923156320130703.3020
INTRODUCTION
The book series Harry Potter enjoy a global popularity 
in recent years. By now most readers are aware of what 
has come to be called the Harry Potter phenomenon. The 
sudden success of the book has resulted in the translations 
of 67 languages. The book is featured with vivid language 
and the wonderful magic world it presents. And both 
children and adults are fascinated by Harry Potter and his 
unusual stories.
with your commander.
Especially not a commander like this one.
Ser Waymar Royce was the youngest son of an ancient house with too many heirs. He 
was a handsome youth of eighteen, grey-eyed and graceful and slender as a knife. 
Mounted on his huge black destrier, the knight towered above Will and Gared on their 
smaller garrons. He wore black leather boots, black woolen pants, black moleskin gloves, 
and a fine supple coat of gleaming black ringmail over layers of black wool and boiled
Education: Study on

In [38]:
for chunk in lotr.invoke("Who is Ser Waymar?"):
    print(chunk.page_content)

the wrong path by making them misunderstand because 
they can hardly judge the effect of the spell from its literal 
meaning. I would tend to think that this spell is to get the 
magical wand to some other places in a peaceful way. 
According to this point we can have a rough conclusion 
that the application of reduplicative is not good to help the 
reader get the immediate understanding of the spell effect. 
They have to refer to the whole story to judge what the 
spell has done to the target.
with your commander.
Especially not a commander like this one.
Ser Waymar Royce was the youngest son of an ancient house with too many heirs. He 
was a handsome youth of eighteen, grey-eyed and graceful and slender as a knife. 
Mounted on his huge black destrier, the knight towered above Will and Gared on their 
smaller garrons. He wore black leather boots, black woolen pants, black moleskin gloves, 
and a fine supple coat of gleaming black ringmail over layers of black wool and boiled
Education

## See this result is too much messy now lets refine it according to the question and overcome the situation of lost in middle

# Now After understanding step by step it create a pipeline for LLM

In [39]:
from langchain_community.document_transformers import (
    EmbeddingsClusteringFilter,
    EmbeddingsRedundantFilter,
    LongContextReorder,
)

from langchain_classic.retrievers.document_compressors import (
    DocumentCompressorPipeline,
)

from langchain_classic.retrievers import (
    ContextualCompressionRetriever,
)

In [40]:
filter = EmbeddingsRedundantFilter(embeddings=bge_embeddings)

reordering = LongContextReorder()

pipeline = DocumentCompressorPipeline(
    transformers=[filter, reordering]
)

compression_retriever_reordered = ContextualCompressionRetriever(
    base_compressor=pipeline,
    base_retriever=lotr,
    search_kwargs={"k": 3, "include_metadata": True},
)

In [41]:
docs1 = compression_retriever_reordered.invoke("Who is harry potter?")

print(docs1)

[_DocumentWithState(metadata={'page': 1, 'producer': 'pypdf', 'creator': 'PyPDF', 'page_label': '2', 'creationdate': '', 'source': 'F:\\Advance RAG\\Data\\got_book.pdf', 'total_pages': 8}, page_content='with your commander.\nEspecially not a commander like this one.\nSer Waymar Royce was the youngest son of an ancient house with too many heirs. He \nwas a handsome youth of eighteen, grey-eyed and graceful and slender as a knife. \nMounted on his huge black destrier, the knight towered above Will and Gared on their \nsmaller garrons. He wore black leather boots, black woolen pants, black moleskin gloves, \nand a fine supple coat of gleaming black ringmail over layers of black wool and boiled', state={'embedded_doc': [0.008487457409501076, -0.008255377411842346, -0.001187325338833034, 0.027800248935818672, -0.021536346524953842, -0.015308336354792118, -0.021682241931557655, 0.018985211849212646, 0.008123782463371754, -0.015270733274519444, 0.040748365223407745, 0.002712150337174535, 0.03

In [43]:
docs = compression_retriever_reordered.invoke("Who is Ser Waymar Royce?")

print(docs)

[_DocumentWithState(metadata={'producer': 'pypdf', 'creationdate': '', 'source': 'F:\\Advance RAG\\Data\\got_book.pdf', 'page': 1, 'total_pages': 8, 'page_label': '2', 'creator': 'PyPDF'}, page_content='with your commander.\nEspecially not a commander like this one.\nSer Waymar Royce was the youngest son of an ancient house with too many heirs. He \nwas a handsome youth of eighteen, grey-eyed and graceful and slender as a knife. \nMounted on his huge black destrier, the knight towered above Will and Gared on their \nsmaller garrons. He wore black leather boots, black woolen pants, black moleskin gloves, \nand a fine supple coat of gleaming black ringmail over layers of black wool and boiled', state={'embedded_doc': [0.008487457409501076, -0.008255377411842346, -0.001187325338833034, 0.027800248935818672, -0.021536346524953842, -0.015308336354792118, -0.021682241931557655, 0.018985211849212646, 0.008123782463371754, -0.015270733274519444, 0.040748365223407745, 0.002712150337174535, 0.03

In [44]:
for doc in docs:
    print(doc.metadata)

{'producer': 'pypdf', 'creationdate': '', 'source': 'F:\\Advance RAG\\Data\\got_book.pdf', 'page': 1, 'total_pages': 8, 'page_label': '2', 'creator': 'PyPDF'}
{'page_label': '2', 'page': 1, 'creationdate': '', 'total_pages': 8, 'creator': 'PyPDF', 'source': 'F:\\Advance RAG\\Data\\got_book.pdf', 'producer': 'pypdf'}
{'producer': 'pypdf', 'creator': 'PyPDF', 'page': 0, 'creationdate': '', 'source': 'F:\\Advance RAG\\Data\\got_book.pdf', 'page_label': '1', 'total_pages': 8}
{'producer': 'pypdf', 'page_label': '3', 'creationdate': '', 'total_pages': 8, 'page': 2, 'creator': 'PyPDF', 'source': 'F:\\Advance RAG\\Data\\got_book.pdf'}
{'creationdate': '', 'source': 'F:\\Advance RAG\\Data\\got_book.pdf', 'producer': 'pypdf', 'creator': 'PyPDF', 'page_label': '5', 'total_pages': 8, 'page': 4}
{'total_pages': 6, 'title': '书籍1.indb', 'moddate': '2014-01-04T17:23:49+08:00', 'source': 'F:\\Advance RAG\\Data\\harry_potter_book.pdf', 'creator': 'Adobe InDesign CS6 (Windows)', 'gts_pdfxversion': 'PDF/

In [45]:
for i, doc in enumerate(docs, start=1):
    print(f"Document {i}")
    print("Metadata:", doc.metadata)
    print("Content:", doc.page_content)
    print("-" * 100)

Document 1
Metadata: {'producer': 'pypdf', 'creationdate': '', 'source': 'F:\\Advance RAG\\Data\\got_book.pdf', 'page': 1, 'total_pages': 8, 'page_label': '2', 'creator': 'PyPDF'}
Content: with your commander.
Especially not a commander like this one.
Ser Waymar Royce was the youngest son of an ancient house with too many heirs. He 
was a handsome youth of eighteen, grey-eyed and graceful and slender as a knife. 
Mounted on his huge black destrier, the knight towered above Will and Gared on their 
smaller garrons. He wore black leather boots, black woolen pants, black moleskin gloves, 
and a fine supple coat of gleaming black ringmail over layers of black wool and boiled
----------------------------------------------------------------------------------------------------
Document 2
Metadata: {'page_label': '2', 'page': 1, 'creationdate': '', 'total_pages': 8, 'creator': 'PyPDF', 'source': 'F:\\Advance RAG\\Data\\got_book.pdf', 'producer': 'pypdf'}
Content: and a fine supple coat of glea

In [47]:
print(len(docs))

10


In [48]:
print(len(docs1))

10


# Load the text generation model(LLM) from huggingface

In [49]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

In [50]:
from transformers import (       #Import AutoTokenizer and AutoModelForCausalLM
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
)

In [51]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  #Define the model name

In [52]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)    #Load the tokenizer

In [53]:
model = AutoModelForCausalLM.from_pretrained(          #Load the model
    MODEL_NAME,     
    device_map="auto",
    torch_dtype="auto",
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.


In [54]:
hf_pipeline = pipeline(               #Create the Hugging Face pipeline
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.7,
    do_sample=True,
)

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [55]:
llm = HuggingFacePipeline(             #Convert it to a LangChain LLM
    pipeline=hf_pipeline
)  

# Using LCEL (LangChain Expression Language)

In [58]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate

In [59]:
template = """
You are a helpful AI assistant.

Answer the question using only the provided context.

Context:
{context}

Question:
{question}

Answer:
"""

prompt = ChatPromptTemplate.from_template(template)

In [60]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [67]:
retrievel_chain = (
    {
        "context": compression_retriever_reordered | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
response = retrievel_chain.invoke(
    "Who is Harry Potter?"
)

print(response)

In [ ]:

query ="who is jon snow?"
results = retrievel_chain.invoke(query)

[transformers] Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In [71]:
print(type(results))

<class 'str'>


In [72]:
print(results)

Human: 
You are a helpful AI assistant.

Answer the question using only the provided context.

Context:
raiders. Each day had been worse than the day that had come before it. Today was the 
worst of all. A cold wind was blowing out of the north, and it made the trees rustle like 
living things. All day, Will had felt as though something were watching him, something 
cold and implacable that loved him not. Gared had felt it too. Will wanted nothing so 
much as to ride hellbent for the safety of the Wall, but that was not a feeling to share 
with your commander.

with your commander.
Especially not a commander like this one.
Ser Waymar Royce was the youngest son of an ancient house with too many heirs. He 
was a handsome youth of eighteen, grey-eyed and graceful and slender as a knife. 
Mounted on his huge black destrier, the knight towered above Will and Gared on their 
smaller garrons. He wore black leather boots, black woolen pants, black moleskin gloves, 
and a fine supple coat of gl

In [77]:
query ="who is Harry Potter?"
results1 = retrievel_chain.invoke(query)

[transformers] Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [78]:
# Get the source documents
source_documents = compression_retriever_reordered.invoke(query)

In [79]:
# Print them
for i, doc in enumerate(source_documents, start=1):
    print(f"\n{'='*60}")
    print(f"Document {i}")
    print(f"Page   : {doc.metadata.get('page')}")
    print(f"Source : {doc.metadata.get('source')}")
    print("-"*60)
    print(doc.page_content)


Document 1
Page   : 1
Source : F:\Advance RAG\Data\got_book.pdf
------------------------------------------------------------
with your commander.
Especially not a commander like this one.
Ser Waymar Royce was the youngest son of an ancient house with too many heirs. He 
was a handsome youth of eighteen, grey-eyed and graceful and slender as a knife. 
Mounted on his huge black destrier, the knight towered above Will and Gared on their 
smaller garrons. He wore black leather boots, black woolen pants, black moleskin gloves, 
and a fine supple coat of gleaming black ringmail over layers of black wool and boiled

Document 2
Page   : 1
Source : F:\Advance RAG\Data\got_book.pdf
------------------------------------------------------------
understand that it was best not to interrupt him when he looked like that. “Tell me again 
what you saw, Will. All the details. Leave nothing out.”
Will had been a hunter before he joined the Night’s Watch. Well, a poacher in truth.

Document 3
Page   : 0
S

In [80]:
print(results1)

Human: 
You are a helpful AI assistant.

Answer the question using only the provided context.

Context:
with your commander.
Especially not a commander like this one.
Ser Waymar Royce was the youngest son of an ancient house with too many heirs. He 
was a handsome youth of eighteen, grey-eyed and graceful and slender as a knife. 
Mounted on his huge black destrier, the knight towered above Will and Gared on their 
smaller garrons. He wore black leather boots, black woolen pants, black moleskin gloves, 
and a fine supple coat of gleaming black ringmail over layers of black wool and boiled

understand that it was best not to interrupt him when he looked like that. “Tell me again 
what you saw, Will. All the details. Leave nothing out.”
Will had been a hunter before he joined the Night’s Watch. Well, a poacher in truth.

Will could see the tightness around Gared’s mouth, the barely suppressed anger in his 
eyes under the thick black hood of his cloak. Gared had spent forty years in the Ni

## Conclusion

This notebook demonstrated a complete multi-source RAG pipeline built around two core LangChain components: MergerRetriever and LongContextReorder.

MergerRetriever (LOTR) unified two completely independent knowledge bases into a single retriever interface. It answered questions about Harry Potter and Game of Thrones through one invoke call by fanning out to both ChromaDB collections simultaneously. This pattern scales to any number of sources without changing any downstream code.

The raw merged output was intentionally noisy, containing up to 10 chunks from both books regardless of which was relevant. The DocumentCompressorPipeline with EmbeddingsRedundantFilter resolved this by removing semantically duplicate chunks. LongContextReorder further improved the result by placing the most relevant chunks at the start and end of the context window where LLMs pay the most attention. This directly addresses the lost-in-the-middle problem that degrades answer quality in naive RAG systems.

The LCEL chain provided a transparent and modifiable pipeline where every transformation step (retrieval, filtering, reordering, prompting, generation, parsing) was explicit and independently inspectable.

The two-embedding-model strategy (bge-large-en for ingestion and retrieval, all-MiniLM-L6-v2 for the redundancy filter) demonstrated a deliberate performance-quality trade-off that is standard in production RAG systems.

Overall this architecture is the recommended approach when building RAG systems over multiple heterogeneous data sources because it keeps each source independently tunable while presenting a single clean interface to the generation layer.